# Comparing the models

Three models trained on **the same 10,000 questions**, with the same context budget (448 state tokens), the same 16 questions per optimizer step, and the same splits:

| run | encoder | how a candidate becomes a vector | decision head | training |
|---|---|---|---|---|
| `cmp-deberta-cls` | DeBERTa-v3-base, one sequence per candidate | `[CLS]` vector | ours: `candidate_mlp` | full fine-tuning |
| `cmp-deberta-mean` | DeBERTa-v3-base, one sequence per candidate | mean of the candidate's tokens | ours: `candidate_mlp` | full fine-tuning |
| `cmp-qwen-bev` | Qwen3-0.6B (first 20 layers), one sequence per question | last token of each candidate | BEV: `bilinear_answer` | LoRA on 12 layers (as in BEV) |

Each model is trained the way it's meant to be trained: DeBERTa fully fine-tuned, Qwen with LoRA as in the original repo. The two DeBERTa runs differ **only** in pooling, which answers the cls versus mean question cleanly.

**Nothing is trained here.** The models come from `bash run_compare.sh`, which trains each config in `configs/compare/`, picks the best checkpoint on validation, and fits per-task temperatures on the calibration split. This notebook loads the best checkpoints and evaluates them once on the same test questions. Test outputs are cached next to each checkpoint, so reruns are fast.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
from safetensors import safe_open

from architectures import get_architecture
from dataset import TASK_NAMES, TASK_TYPES, QuestionDataset, load_questions
from inference import answer, get_device
from logger import load_checkpoint
from metrics import compute_metrics
from notebook_utils import cached_logits, calibration_curves, free_memory, latest_run, plot_reliability, read_metrics

PREFIX = os.environ.get("NOTEBOOK_RUN_PREFIX", "cmp")  # run-name prefix used by run_compare.sh
RUNS = {
    "DeBERTa · cls": f"{PREFIX}-deberta-cls",
    "DeBERTa · mean": f"{PREFIX}-deberta-mean",
    "Qwen · BEV": f"{PREFIX}-qwen-bev",
}
NUM_TEST = int(os.environ.get("NOTEBOOK_NUM_TEST", 2000))
EVAL_BATCH_SIZE = 8
COLORS = {"DeBERTa · cls": "C0", "DeBERTa · mean": "C1", "Qwen · BEV": "C2"}

device = get_device()
pd.options.display.float_format = "{:.4f}".format
run_dirs = {label: latest_run(name) for label, name in RUNS.items()}
missing = [RUNS[label] for label, r in run_dirs.items() if r is None]
assert not missing, f"no finished run for {missing}: run `bash run_compare.sh` first"
for label, r in run_dirs.items():
    print(f"{label:<16} {r}")

## 1. What was trained

In [ ]:
def describe(run_dir):
    config = json.loads(json.dumps(__import__("yaml").safe_load((run_dir / "config.yaml").read_text())))
    m, lora = config["model"], config["lora"]
    best = run_dir / "checkpoints" / "best"
    with safe_open(best / "weights.safetensors", "pt") as f:
        trainable = sum(torch.Size(f.get_slice(k).get_shape()).numel() for k in f.keys())
    calibration = json.loads((best / "calibration.json").read_text()) if (best / "calibration.json").exists() else {}
    summary = json.loads((run_dir / "summary.json").read_text())
    return {
        "encoder": m["name"].split("/")[-1],
        "pooling": m.get("pooling", "last token"),
        "head": config["head"].get("type", "bilinear_answer"),
        "fine-tuning": "full" if m.get("full_finetune") else (f"LoRA, {lora['last_k_layers']} layers" if lora["last_k_layers"] else "frozen"),
        "trainable params (M)": trainable / 1e6,
        "train questions": config["data"]["max_train_questions"],
        "training time (min)": summary["duration_sec"] / 60,
        "best step": json.loads((best / "meta.json").read_text())["step"],
        "best validation acc": summary.get("best_test_accuracy"),
        "temperatures": {k: round(v, 2) for k, v in calibration.get("temperatures", {}).items()},
    }

overview = pd.DataFrame({label: describe(r) for label, r in run_dirs.items()}).T
overview

## 2. Training curves

Every model takes 16 questions per optimizer step, so the x-axes line up. Validation is the validation split (1,000 questions). Dots with a black edge mark each model's best checkpoint.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for label, run_dir in run_dirs.items():
    h, c = read_metrics(run_dir), COLORS[label]
    best = overview.loc[label, "best step"]
    axes[0].plot([r["step"] for r in h["train"]], [r["loss"] for r in h["train"]], c=c, alpha=0.8, label=label)
    for ax, key in [(axes[1], "loss"), (axes[2], "accuracy")]:
        steps, values = [r["step"] for r in h["validation"]], [r[key] for r in h["validation"]]
        ax.plot(steps, values, "o-", c=c, label=label)
        ax.plot([best], [values[steps.index(best)]], "o", c=c, mec="black", mew=1.5, ms=9)
for ax, title in zip(axes, ["train loss", "validation loss (NLL)", "validation accuracy"]):
    ax.set(title=title, xlabel="optimizer step (16 questions)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Test evaluation

The same `NUM_TEST` test questions for every model, in the same order, so results can be compared question by question. Each model uses its own calibrated temperatures from `calibration.json`.

In [ ]:
test_q = load_questions("test", NUM_TEST, seed=0)
results = {}
for label, run_dir in run_dirs.items():
    best = run_dir / "checkpoints" / "best"
    network, config = load_checkpoint(best, device)
    arch, d = get_architecture(config), config["data"]
    loader = torch.utils.data.DataLoader(QuestionDataset(test_q, arch.encode_example, d["max_state_tokens"], d["max_choice_tokens"]),
                                         batch_size=EVAL_BATCH_SIZE, collate_fn=arch.collate_fn)
    out = cached_logits(network, best, "test", loader, device)
    out["temperature"] = network.temperature
    out["raw"] = compute_metrics(out["logits"], out["labels"], out["task_types"])
    out["calibrated"] = compute_metrics(out["logits"], out["labels"], out["task_types"], network.temperature)
    results[label] = out
    print(f"{label:<16} accuracy {out['calibrated']['accuracy']:.4f} | {out['seconds_per_question'] * 1000:.0f} ms/question")
    del network
    free_memory(device)

labels, task_types = results[next(iter(results))]["labels"], results[next(iter(results))]["task_types"]
assert all(torch.equal(r["labels"], labels) for r in results.values()), "models were not evaluated on the same questions"

In [ ]:
columns = ["accuracy", "accuracy_choice", "accuracy_noul", "accuracy_score", "loss", "nll_choice", "nll_noul", "nll_score",
           "auc_noul", "brier_noul", "mae_score", "ece_choice", "ece_noul", "ece_score"]
table = pd.DataFrame({label: {k: r["calibrated"].get(k) for k in columns} for label, r in results.items()}).T
table["ms / question"] = [r["seconds_per_question"] * 1000 for r in results.values()]
print(f"Test set, {len(labels)} questions, calibrated probabilities ({ {TASK_NAMES[t]: int((task_types == t).sum()) for t in TASK_NAMES} })")
higher_is_better = {"accuracy", "accuracy_choice", "accuracy_noul", "accuracy_score", "auc_noul"}
table.style.format("{:.4f}").format("{:.0f}", subset=["ms / question"]).highlight_max(
    subset=[c for c in columns if c in higher_is_better], color="#c8e6c9").highlight_min(
    subset=[c for c in columns if c not in higher_is_better] + ["ms / question"], color="#c8e6c9")

**Effect of calibration**: NLL before and after temperature scaling. Accuracy is identical by construction.

In [ ]:
pd.DataFrame({label: {"NLL, T = 1": r["raw"]["loss"], "NLL, calibrated": r["calibrated"]["loss"],
                      "Brier noul, T = 1": r["raw"].get("brier_noul"), "Brier noul, calibrated": r["calibrated"].get("brier_noul"),
                      "temperatures": {k: round(v, 2) for k, v in r["temperature"].items()} if isinstance(r["temperature"], dict) else r["temperature"]}
              for label, r in results.items()}).T

## 4. Are the differences real?

On 2,000 questions, a 1-point accuracy gap can be noise. A **paired bootstrap** resamples the test questions 2,000 times and recomputes the accuracy difference on each resample, using the same questions for both models. If the 95% interval excludes 0, the difference is unlikely to be luck.

In [ ]:
def hits(label, mask=None):
    r = results[label]
    h = r["logits"].argmax(-1) == r["labels"]
    return h if mask is None else h[mask]

def paired_bootstrap(a, b, n=2000, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(len(a), (n, len(a)), generator=g)
    diffs = a.float()[idx].mean(1) - b.float()[idx].mean(1)
    return (a.float().mean() - b.float().mean()).item(), diffs.quantile(0.025).item(), diffs.quantile(0.975).item()

best_deberta = max(["DeBERTa · cls", "DeBERTa · mean"], key=lambda l: results[l]["calibrated"]["accuracy"])
pairs = [("DeBERTa · mean", "DeBERTa · cls"), (best_deberta, "Qwen · BEV")]
rows = []
for a, b in pairs:
    for task in [None, *TASK_TYPES]:
        mask = None if task is None else task_types == TASK_TYPES[task]
        diff, lo, hi = paired_bootstrap(hits(a, mask), hits(b, mask))
        rows.append({"comparison": f"{a}  −  {b}", "questions": "all" if task is None else task,
                     "accuracy difference": diff, "95% CI low": lo, "95% CI high": hi,
                     "significant": "yes" if lo > 0 or hi < 0 else "no"})
pd.DataFrame(rows).set_index(["comparison", "questions"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
tasks = ["overall", *TASK_TYPES]
width = 0.8 / len(results)
for i, (label, r) in enumerate(results.items()):
    accs, errs = [], []
    for task in tasks:
        mask = None if task == "overall" else task_types == TASK_TYPES[task]
        h = hits(label, mask).float()
        g = torch.Generator().manual_seed(0)
        boots = h[torch.randint(len(h), (1000, len(h)), generator=g)].mean(1)
        accs.append(h.mean().item())
        errs.append([[h.mean().item() - boots.quantile(0.025).item()], [boots.quantile(0.975).item() - h.mean().item()]])
    xs = [j + (i - (len(results) - 1) / 2) * width for j in range(len(tasks))]
    axes[0].bar(xs, accs, width, yerr=torch.tensor(errs).squeeze(-1).T.numpy(), capsize=3, color=COLORS[label], label=label)
axes[0].set_xticks(range(len(tasks)), tasks)
axes[0].set(ylim=(0, 1), title="test accuracy (95% bootstrap interval)"); axes[0].legend(fontsize=8)

for label, r in results.items():
    p, y, *_ = calibration_curves(r["logits"], r["labels"], r["task_types"], r["temperature"])["noul"]
    plot_reliability(axes[1], p, y, "predicted P(yes)", "observed fraction yes", label=label, show_counts=False, c=COLORS[label])
axes[1].set_title("yes/no calibration on test (calibrated)"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Where do the best DeBERTa and Qwen disagree?

Test questions that one model gets right and the other gets wrong, with each model's probability for the correct answer.

In [ ]:
a, b = best_deberta, "Qwen · BEV"
probs = {l: torch.softmax(results[l]["logits"] / torch.tensor([results[l]["temperature"][TASK_NAMES[int(t)]] if isinstance(results[l]["temperature"], dict) else results[l]["temperature"] for t in task_types])[:, None], -1) for l in (a, b)}
only_a = (hits(a) & ~hits(b)).nonzero().flatten()
only_b = (hits(b) & ~hits(a)).nonzero().flatten()
print(f"{a} right, {b} wrong: {len(only_a)} questions | {b} right, {a} wrong: {len(only_b)} questions\n")

def show(i):
    row, y = test_q[int(i)], int(labels[i])
    print(f"[{TASK_NAMES[row['task_type']]}] {row['instructions']}")
    print("   state:", row["state"][:220].replace("\n", " "), "…")
    for c, choice in enumerate(row["choices"]):
        pa, pb = probs[a][i, c].item(), probs[b][i, c].item()
        print(f"   {'→' if c == y else ' '} {choice[:60]:<62} {a.split(' ·')[0]} {pa:5.2f}   Qwen {pb:5.2f}")
    print()

for i in only_a[:3]: show(i)
print("-" * 100 + "\n")
for i in only_b[:3]: show(i)

## 6. The same questions, every model

Edit the questions and rerun. Each model is loaded in turn.

In [ ]:
state = ("Hi, I was charged twice for my March invoice (order #4821). I have contacted you three times already "
         "and nobody answers. If this is not refunded by Friday I will cancel my subscription.")
questions = [
    {"type": "choice", "instructions": "Which department should handle this ticket?",
     "criteria": {"Billing": "payments, invoices, refunds", "Technical": "bugs, outages, login problems",
                  "Sales": "new purchases, upgrades, pricing questions"}},
    {"type": "noul", "instructions": "Is the customer asking for a refund?"},
    {"type": "noul", "instructions": "Is the customer asking for a new product?"},
    {"type": "score", "instructions": "How frustrated is the customer?",
     "criteria": ["calm", "slightly annoyed", "annoyed", "very frustrated", "furious"]},
]

def short(result, q):
    if q["type"] == "noul":
        return f"P(yes) = {result['noul']:.2f}"
    if q["type"] == "score":
        return f"E[score] = {result['score']:.2f}"
    return ", ".join(f"{k} {p:.2f}" for k, p in result["probabilities"].items())

answers = {}
for label, run_dir in run_dirs.items():
    network, config = load_checkpoint(run_dir / "checkpoints" / "best", device)
    answers[label] = [short(answer(network, config, state, q, device), q) for q in questions]
    del network
    free_memory(device)

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(answers, index=[f"[{q['type']}] {q['instructions']}" for q in questions])

## 7. Conclusions

*Written after the runs finished.*